# 02 - Exercise 2 Walkthrough

This notebook is the learning path for Exercise 2.

Recommended order:

1. run `00_setup_and_load_data.ipynb`
2. run `01_data_eda_and_assumptions.ipynb`
3. run this walkthrough to understand the logic step by step
4. run `03_exercise_2_production_solution.ipynb` to produce the final files quickly

Target question:

> What are the top 10 songs played in the top 50 longest sessions by track count?


## Functions used in this notebook

### `create_spark_session(...)`
- **Description:** Starts Spark with the same settings used by the rest of the project.
- **Input:** Notebook app name and optional Spark parameters.
- **Output:** Configured `SparkSession`.
- **Why this operation is selected for efficiency:** The walkthrough runs on the same execution profile as the final solution, so there is no gap between the pedagogical path and the scalable path.

### `load_lastfm_events(...)`
- **Description:** Loads the normalized play events, defaulting to Parquet if it already exists.
- **Input:** Spark session and project root.
- **Output:** Spark DataFrame of normalized events.
- **Why this operation is selected for efficiency:** It avoids repeating raw TSV parsing during the exploratory walkthrough.

### `compute_sessions(events_df, gap_minutes=20)`
- **Description:** Assigns a session ID to each play event by comparing each play with the previous play for the same user.
- **Input:** Event-level DataFrame and the 20-minute session gap rule.
- **Output:** Event-level DataFrame enriched with session columns such as `session_id`, `gap_seconds`, and `is_new_session`.
- **Why this operation is selected for efficiency:** This row-level version is excellent for teaching and validation because it makes the session rule visible on real events before we move to the more scalable aggregation path.

### `summarize_sessions_from_events(events_df, gap_minutes=20)`
- **Description:** Builds session summaries directly from the event table with Spark `session_window`.
- **Input:** Event-level DataFrame and the session gap rule.
- **Output:** One row per session with track count, start/end timestamps, duration, and generated `session_id`.
- **Why this operation is selected for efficiency:** It avoids materializing a large sessionized event table for the full dataset and therefore scales better for bigger inputs.

### `top_longest_sessions_by_track_count(...)`
- **Description:** Sorts sessions by track count and returns the top 50 required by the exercise.
- **Input:** Session summary DataFrame and the number of sessions to keep.
- **Output:** Top-session DataFrame.
- **Why this operation is selected for efficiency:** It narrows the problem early, so later joins and aggregations only touch the small set of winning sessions.

### `top_songs_from_longest_sessions(...)`
- **Description:** Computes the top songs inside the selected top sessions.
- **Input:** Event-level DataFrame plus the session summary DataFrame.
- **Output:** Ranked song table with play counts and supporting metadata.
- **Why this operation is selected for efficiency:** The function broadcasts the small top-session set back to the event table, which is much cheaper than carrying all sessions through every step.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


WindowsPath('C:/Users/GonzaloFigueroa/Documents/Private/BME/coding challenge')

In [2]:
from pyspark.sql import functions as F

from src.spark_utils import create_spark_session
from src.sessionization import (
    ensure_lastfm_dataset_available,
    compute_sessions,
    load_lastfm_events,
    resolve_lastfm_input_path,
    stage_lastfm_events_to_parquet,
    summarize_sessions_from_events,
    top_longest_sessions_by_track_count,
    top_songs_from_longest_sessions,
)


In [3]:
spark = create_spark_session(app_name='02-exercise-2-walkthrough')
spark


In [4]:
raw_input_path = ensure_lastfm_dataset_available(PROJECT_ROOT)
parquet_path = stage_lastfm_events_to_parquet(spark, PROJECT_ROOT)
print(f'Raw dataset path: {raw_input_path}')
print(f'Prepared Parquet path: {parquet_path}')


Raw dataset path: C:\Users\GonzaloFigueroa\Documents\Private\BME\coding challenge\data\raw\lastfm-dataset-1K\lastfm-dataset-1K\userid-timestamp-artid-artname-traid-traname.tsv
Prepared Parquet path: C:\Users\GonzaloFigueroa\Documents\Private\BME\coding challenge\data\processed\lastfm_events_parquet


In [5]:
events_df = load_lastfm_events(spark, PROJECT_ROOT)
events_df.show(5, truncate=False)


+-----------+-------------------+------------------------------------+-------------+------------------------------------+----------------+------------------------------------+
|user_id    |started_at         |artist_id                           |artist_name  |track_id                            |track_name      |song_id                             |
+-----------+-------------------+------------------------------------+-------------+------------------------------------+----------------+------------------------------------+
|user_000013|2006-12-22 18:07:11|835a6d9c-fea0-4a71-ae52-9c4da946433a|Styx         |ea4bd0c5-c323-4d21-9c78-1bc9c46e5da8|Mr. Roboto      |ea4bd0c5-c323-4d21-9c78-1bc9c46e5da8|
|user_000013|2006-12-22 18:12:00|NULL                                |Subwoofer    |NULL                                |Vaporize        |Vaporize                            |
|user_000013|2006-12-22 18:16:36|24786816-025f-49f4-9787-4945a3311f96|The Flashbulb|2b02f1d0-33f6-4635-9316-44ec4dcd483d

In [6]:
events_df.agg(
    F.count('*').alias('total_rows'),
    F.countDistinct('user_id').alias('distinct_users'),
    F.min('started_at').alias('min_started_at'),
    F.max('started_at').alias('max_started_at'),
).show(truncate=False)


+----------+--------------+-------------------+-------------------+
|total_rows|distinct_users|min_started_at     |max_started_at     |
+----------+--------------+-------------------+-------------------+
|19150867  |992           |2005-02-14 00:00:07|2013-09-29 18:32:04|
+----------+--------------+-------------------+-------------------+



## Understand the session rule on one sample user

The next cells show the row-level logic on a small ordered sample. This is the human-readable explanation path.


In [7]:
sample_user_row = (
    events_df.groupBy('user_id')
    .count()
    .orderBy(F.desc('count'), F.asc('user_id'))
    .first()
)

sample_user_id = sample_user_row['user_id']
print(f'Sample user selected: {sample_user_id}')
print(f"Number of plays for that user: {sample_user_row['count']}")


Sample user selected: user_000949
Number of plays for that user: 183103


In [8]:
sample_user_events_df = (
    events_df.filter(F.col('user_id') == sample_user_id)
    .orderBy('started_at')
    .limit(40)
)

sample_user_events_df.select('user_id', 'started_at', 'artist_name', 'track_name').show(40, truncate=False)


+-----------+-------------------+---------------------+---------------------------------+
|user_id    |started_at         |artist_name          |track_name                       |
+-----------+-------------------+---------------------+---------------------------------+
|user_000949|2005-05-30 06:15:32|Pedro The Lion       |Bad Diary Days                   |
|user_000949|2005-05-30 06:19:54|Daft Punk            |Technologic                      |
|user_000949|2005-05-30 06:24:17|Radiohead            |Everything In Its Right Place    |
|user_000949|2005-05-30 06:28:21|The Rolling Stones   |Paint It Black                   |
|user_000949|2005-05-30 06:31:33|M83                  |Run Into Flowers                 |
|user_000949|2005-05-30 06:40:11|The Faint            |In Concert                       |
|user_000949|2005-05-30 06:41:53|Josh Rouse           |Directions                       |
|user_000949|2005-05-30 06:43:56|Interpol             |Obstacle 1                       |
|user_0009

In [9]:
sample_user_sessionized_df = compute_sessions(sample_user_events_df, gap_minutes=20)

sample_user_sessionized_df.select(
    'user_id', 'started_at', 'track_name', 'previous_started_at', 'gap_seconds', 'is_new_session', 'session_id'
).show(40, truncate=False)


+-----------+-------------------+---------------------------------+-------------------+-----------+--------------+--------------------+
|user_id    |started_at         |track_name                       |previous_started_at|gap_seconds|is_new_session|session_id          |
+-----------+-------------------+---------------------------------+-------------------+-----------+--------------+--------------------+
|user_000949|2005-05-30 06:15:32|Bad Diary Days                   |NULL               |NULL       |1             |user_000949-00000001|
|user_000949|2005-05-30 06:19:54|Technologic                      |2005-05-30 06:15:32|262        |0             |user_000949-00000001|
|user_000949|2005-05-30 06:24:17|Everything In Its Right Place    |2005-05-30 06:19:54|263        |0             |user_000949-00000001|
|user_000949|2005-05-30 06:28:21|Paint It Black                   |2005-05-30 06:24:17|244        |0             |user_000949-00000001|
|user_000949|2005-05-30 06:31:33|Run Into Flower

## Switch to the scalable full-dataset session summary

The production path does not materialize every row with a session ID. It builds one row per session directly from the events table using Spark `session_window`.


In [10]:
session_summary_df = summarize_sessions_from_events(events_df, gap_minutes=20)
session_summary_df.orderBy(F.desc('track_count'), F.asc('session_start')).show(20, truncate=False)


+-----------+-----------+-------------------+-------------------+--------------+--------------------+------------------------+------------------------+
|user_id    |track_count|session_start      |session_end        |session_number|session_id          |session_duration_seconds|session_duration_minutes|
+-----------+-----------+-------------------+-------------------+--------------+--------------------+------------------------+------------------------+
|user_000949|5360       |2006-02-12 17:49:31|2006-02-27 11:29:37|151           |user_000949-00000151|1273206                 |21220.1                 |
|user_000544|5350       |2007-02-12 13:03:52|2007-02-23 00:51:08|75            |user_000544-00000075|906436                  |15107.266666666666      |
|user_000949|4956       |2005-12-09 08:26:38|2005-12-18 04:40:04|139           |user_000949-00000139|764006                  |12733.433333333332      |
|user_000949|4705       |2007-05-01 02:41:15|2007-05-14 00:05:52|559           |user_000

In [11]:
session_summary_df.agg(
    F.count('*').alias('total_sessions'),
    F.avg('track_count').alias('avg_tracks_per_session'),
    F.max('track_count').alias('max_tracks_in_one_session'),
).show(truncate=False)


+--------------+----------------------+-------------------------+
|total_sessions|avg_tracks_per_session|max_tracks_in_one_session|
+--------------+----------------------+-------------------------+
|1041883       |18.381014950815015    |5360                     |
+--------------+----------------------+-------------------------+



In [12]:
top_50_sessions_df = top_longest_sessions_by_track_count(
    session_summary_df=session_summary_df,
    top_session_count=50,
)

top_50_sessions_df.show(50, truncate=False)


+-----------+-----------+-------------------+-------------------+--------------+--------------------+------------------------+------------------------+
|user_id    |track_count|session_start      |session_end        |session_number|session_id          |session_duration_seconds|session_duration_minutes|
+-----------+-----------+-------------------+-------------------+--------------+--------------------+------------------------+------------------------+
|user_000949|5360       |2006-02-12 17:49:31|2006-02-27 11:29:37|151           |user_000949-00000151|1273206                 |21220.1                 |
|user_000544|5350       |2007-02-12 13:03:52|2007-02-23 00:51:08|75            |user_000544-00000075|906436                  |15107.266666666666      |
|user_000949|4956       |2005-12-09 08:26:38|2005-12-18 04:40:04|139           |user_000949-00000139|764006                  |12733.433333333332      |
|user_000949|4705       |2007-05-01 02:41:15|2007-05-14 00:05:52|559           |user_000

In [13]:
top_10_songs_df = top_songs_from_longest_sessions(
    events_df,
    session_summary_df=session_summary_df,
    top_session_count=50,
    top_song_count=10,
)

top_10_songs_df.show(10, truncate=False)


+-------------------------+-------------------------------------+----------+-------------+------------------+
|artist_name              |track_name                           |play_count|session_count|distinct_track_ids|
+-------------------------+-------------------------------------+----------+-------------+------------------+
|Cake                     |Jolene                               |1214      |12           |1                 |
|The Knife                |Heartbeats                           |868       |2            |1                 |
|Jeff Buckley & Gary Lucas|How Long Will It Take                |726       |2            |1                 |
|Broken Social Scene      |Anthems For A Seventeen Year Old Girl|659       |6            |1                 |
|Elliott Smith            |St. Ides Heaven                      |646       |6            |1                 |
|The Killers              |Bonus Track                          |634       |12           |0                 |
|2Pac     

In [14]:
top_10_songs_pd = top_10_songs_df.toPandas()
top_10_songs_pd


,artist_name,track_name,play_count,session_count,distinct_track_ids
0,Cake,Jolene,1214,12,1
1,The Knife,Heartbeats,868,2,1
2,Jeff Buckley & Gary Lucas,How Long Will It Take,726,2,1
3,Broken Social Scene,Anthems For A Seventeen Year Old Girl,659,6,1
4,Elliott Smith,St. Ides Heaven,646,6,1
5,The Killers,Bonus Track,634,12,0
6,2Pac,Starin' Through My Rear View,617,12,1
7,The Rolling Stones,Beast Of Burden,613,3,1
8,Everclear,The Swing,604,15,1
9,Kanye West,See You In My Nightmares,536,3,1


In [15]:
for idx, row in top_10_songs_pd.iterrows():
    print(f"{idx + 1}. {row['artist_name']} - {row['track_name']} (plays={row['play_count']}, sessions={row['session_count']})")


1. Cake - Jolene (plays=1214, sessions=12)
2. The Knife - Heartbeats (plays=868, sessions=2)
3. Jeff Buckley & Gary Lucas - How Long Will It Take (plays=726, sessions=2)
4. Broken Social Scene - Anthems For A Seventeen Year Old Girl (plays=659, sessions=6)
5. Elliott Smith - St. Ides Heaven (plays=646, sessions=6)
6. The Killers - Bonus Track (plays=634, sessions=12)
7. 2Pac - Starin' Through My Rear View (plays=617, sessions=12)
8. The Rolling Stones - Beast Of Burden (plays=613, sessions=3)
9. Everclear - The Swing (plays=604, sessions=15)
10. Kanye West - See You In My Nightmares (plays=536, sessions=3)


When the logic is clear, move to `03_exercise_2_production_solution.ipynb` to generate the final answer files quickly.


In [ ]:
spark.stop()
